# Lanzar simulaciones RRAM

Notebook adaptado a la **arquitectura init → exec → plot**.

Cada fase es independiente:
- **init**: pre-genera `Init_data/init_state_*.npz` para todas las sims del CSV.
- **exec**: ejecuta el ciclo SET → RESET de una simulación. Persiste `Data_*.npz` + `sim_metadata.json` en `Results/Simulation_{N}/`.
- **plot**: lee del disco y genera figuras. **No** re-ejecuta exec.

Lanzamiento via subcomando `python -m RRAM <init|exec|plot|all>`.

Logging: nivel global por env var `RRAM_LOG_LEVEL=DEBUG|INFO|WARNING`. Cada subprocess escribe `logs/log_simulacion_{N}.log`.

In [ ]:
%load_ext autoreload
%autoreload 2

import concurrent.futures
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from RRAM import simulation_config

ruta_raiz = Path.cwd()
print('Ruta raiz del proyecto:', ruta_raiz)

## 1. Estructura de carpetas

In [ ]:
def setup_project_structure():
    """Crea todas las carpetas necesarias antes de ejecutar las simulaciones."""
    folders = [
        'Init_data',
        'Datos_Experimentales/Ciclos_Experimentales',
        'Results',
        'Results/Figures',
        'logs',
    ]
    for folder in folders:
        Path(folder).mkdir(parents=True, exist_ok=True)
        print(f'OK carpeta: {folder}/')

    exp_files = [
        'Datos_Experimentales/Ciclos_Experimentales/Cycle_p_1000.txt',
        'Datos_Experimentales/Ciclos_Experimentales/Cycle_n_1000.txt',
    ]
    print('\nArchivos que DEBES proporcionar manualmente:')
    for f in exp_files:
        marker = 'OK' if os.path.exists(f) else 'FALTA'
        print(f'  [{marker}] {f}')


setup_project_structure()

In [ ]:
def safe_reset_folder(folder_path: Path | str) -> None:
    """Borra y recrea una carpeta, evitando rutas peligrosas."""
    folder_path = str(folder_path)
    proteccion = ['c:/users/usuario', 'c:\\users\\usuario', 'c:/users', 'c:\\users', '/']
    if folder_path.strip().lower() in proteccion:
        print(f'ADVERTENCIA: carpeta protegida, no se borra: {folder_path}')
        return
    try:
        if os.path.exists(folder_path):
            shutil.rmtree(folder_path)
        os.makedirs(folder_path)
    except Exception as e:
        print(f'Error al resetear {folder_path}: {e}')


# Limpiar resultados previos
carpeta_results = ruta_raiz / 'Results'
if carpeta_results.exists():
    shutil.rmtree(carpeta_results)
carpeta_results.mkdir(parents=True, exist_ok=True)
(carpeta_results / 'Figures').mkdir(parents=True, exist_ok=True)

# Limpiar logs previos
log_dir = ruta_raiz / 'logs'
log_dir.mkdir(exist_ok=True)
for f in log_dir.iterdir():
    if f.is_file():
        try:
            f.unlink()
        except Exception as e:
            print(f'No se pudo eliminar {f}: {e}')

print('Estructura limpia.')

## 2. Generación de parámetros (CSV → Init_data)

Construye `simulation_parameters.csv` y `simulation_constants.csv` mediante el barrido de parámetros.

In [ ]:
manager = simulation_config.ConfigManager()
carpeta_init = 'Init_data'

# Bases de Montecarlo (replicar la fuerza bruta original)
factors_set = [1]
factors_reset = [1]
E_a_base = 1.09
E_r_base = 1.55

# Diccionario de barrido (Grid Search)
parametros_barrido = {
    'device_size_y': [15e-9],
    'ohm_resistence_set': [4.1],
    'ohm_resistence_reset': [4.1]*14,
    'grosor_filamento': [[3, 1], [1, 0]],
    'generation_energy': [round(E_a_base * f, 3) for f in factors_set],
    'recombination_energy': [round(E_r_base * f, 3) for f in factors_reset],
}

manager.add_sweep(sweep_params=parametros_barrido)
manager.export_to_init_data(carpeta_init)

num_simulaciones = len(manager.simulations)
print(f'\nGeneradas {num_simulaciones} configuraciones en {carpeta_init}/.')

## 3. INIT — pre-generar estados iniciales

Equivale al antiguo `Init_simulation.py`. Lee el CSV recién creado y deja `Init_data/init_state_{i}.npz` para cada simulación.

In [ ]:
resultado = subprocess.run(
    [sys.executable, '-m', 'RRAM', 'init'],
    cwd=str(ruta_raiz),
    capture_output=True, text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print('STDERR:', resultado.stderr)
    raise RuntimeError('init falló')

## 4. EXEC — lanzamiento paralelo del ciclo SET → RESET

Cada simulación es un subprocess `python -m RRAM exec <num>`. La salida ya **no** se redirige: `setup_logging(num+1)` dentro del subprocess escribe directamente `logs/log_simulacion_{N+1}.log`.

Variables de control:
- `RRAM_LOG_LEVEL` por env var (`DEBUG`/`INFO`/`WARNING`).
- `--num-filamentos N` se pasa al CLI.

In [ ]:
# Configuración de ejecución
guardar_datos = False
num_filamentos = 2
log_level = os.environ.get('RRAM_LOG_LEVEL', 'INFO')   # cambia a 'DEBUG' para trazar variables
num_procesadores = max(int(0.2 * (os.cpu_count() or 10)), 1)

print(f'Lanzando {num_simulaciones} simulaciones con {num_procesadores} CF. RRAM_LOG_LEVEL={log_level}')


def ejecutar_simulacion(num_simulacion: int) -> int:
    env = os.environ.copy()
    env['RRAM_LOG_LEVEL'] = log_level

    cmd = [
        sys.executable, '-m', 'RRAM', 'exec',
        str(num_simulacion),
        '--num-filamentos', str(num_filamentos),
    ]
    print(f'  Iniciando exec sim={num_simulacion+1}')
    completed = subprocess.run(cmd, cwd=str(ruta_raiz), env=env)
    if completed.returncode != 0:
        print(f'  AVISO: sim {num_simulacion+1} terminó con código {completed.returncode}')
    return completed.returncode


start_time = time.time()
with concurrent.futures.ThreadPoolExecutor(max_workers=num_procesadores) as executor:
    list(executor.map(ejecutar_simulacion, range(num_simulaciones)))
elapsed_h = (time.time() - start_time) / 3600

print(f'\nTodas las simulaciones han terminado.')
print(f'Tiempo total: {elapsed_h:.2f} h ({elapsed_h*3600:.1f} s)')

In [ ]:
# Se avisa al usuario que ya ha acabado la simulación
import requests
import urllib.parse


def enviar_notificacion_whatsapp(n_sims, tiempo_total):
    telefono = "+34689866027"
    apikey = "2486810"

    # f-string con los datos de tu notebook
    mensaje = f"✅ Simulaciones RRAM completadas. Procesadas: {n_sims} en {tiempo_total:.2f}h."

    # Es vital codificar el mensaje para evitar errores en la API
    mensaje_url = urllib.parse.quote(mensaje)
    url = f"https://api.callmebot.com/whatsapp.php?phone={telefono}&text={mensaje_url}&apikey={apikey}"

    try:
        # Definimos un timeout para que el notebook no se quede "colgado" si la API tarda en responder
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            print("Petición enviada a CallMeBot.")
    except Exception as e:
        print(f"Error de conexión: {e}")

enviar_notificacion_whatsapp(num_simulaciones, elapsed_h)

## 5. PLOT — generar figuras leyendo del disco (NO re-ejecuta)

Independiente de la celda anterior. Funciona también en una sesión nueva donde solo tengas las carpetas `Results/Simulation_{N}/`.

In [ ]:
# Plot vía CLI (subprocess paralelo opcional)
def plotear_simulacion(num_simulacion: int) -> int:
    # `exec` guarda en Simulation_{num+1}/, así que `plot` recibe num+1.
    n_save = num_simulacion + 1
    cmd = [sys.executable, '-m', 'RRAM', 'plot', str(n_save)]
    completed = subprocess.run(cmd, cwd=str(ruta_raiz))
    if completed.returncode != 0:
        print(f'  Plot falló para sim {n_save}')
    return completed.returncode


for n in range(num_simulaciones):
    plotear_simulacion(n)

print('\nFiguras generadas.')

### Plot programático (sin subprocess)

Útil cuando quieres replotear **sin** salir del notebook ni re-ejecutar el ciclo:

In [ ]:
from RRAM.plot_results import plot_results
from RRAM.logging_config import setup_logging

# Replot sin subprocess (más rápido para iterar sobre estilos de gráfico)
setup_logging(num_simulation=None, to_console=True)   # log a stderr en el notebook

for n in range(num_simulaciones):
    plot_results(num_simulation=n + 1)

## 6. Representación

In [ ]:
from RRAM.plot_results import plot_estados

# Estado de todas las sims, todas las fases

plot_estados(plot_types=["thermal"], phases=["pp_set"], sim_indices=list(range(1, 2)))
